In [ ]:
import os, sys

# Run as if from the project root, regardless of where this notebook lives
if os.path.basename(os.getcwd()) == "notebook":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

In [ ]:
import numpy as np
import pickle
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression


FEATURES_FILE = "./extracted_features/en_us_features.pkl"
LAYER = 6
FEATURE_TO_PLOT = "voicing"   # or "manner"



from collections import defaultdict, Counter
import re

def simple_g2p(text):
    text = text.lower().strip()
    text = re.sub(r"[^a-z ]", "", text)

    consonants = set("bcdfghjklmnpqrstvwxyz")
    vowels = set("aeiou")

    phonemes = []

    for ch in text:
        if ch in consonants:
            phonemes.append(ch)
        elif ch in vowels:
            phonemes.append(ch.upper())

    return phonemes


PHONEME_FEATURES = {
    'p': {'voicing': 'voiceless', 'manner': 'stop'},
    'b': {'voicing': 'voiced', 'manner': 'stop'},
    't': {'voicing': 'voiceless', 'manner': 'stop'},
    'd': {'voicing': 'voiced', 'manner': 'stop'},
    'k': {'voicing': 'voiceless', 'manner': 'stop'},
    'g': {'voicing': 'voiced', 'manner': 'stop'},
    'f': {'voicing': 'voiceless', 'manner': 'fricative'},
    'v': {'voicing': 'voiced', 'manner': 'fricative'},
    's': {'voicing': 'voiceless', 'manner': 'fricative'},
    'z': {'voicing': 'voiced', 'manner': 'fricative'},
    'm': {'voicing': 'voiced', 'manner': 'nasal'},
    'n': {'voicing': 'voiced', 'manner': 'nasal'},
    'l': {'voicing': 'voiced', 'manner': 'liquid'},
    'r': {'voicing': 'voiced', 'manner': 'liquid'},
}

def extract_features(text):
    phonemes = simple_g2p(text)
    if not phonemes:
        return None

    feats = defaultdict(list)

    for p in phonemes:
        if p in PHONEME_FEATURES:
            for k, v in PHONEME_FEATURES[p].items():
                feats[k].append(v)

    if not feats:
        return None

    return {k: Counter(v).most_common(1)[0][0] for k, v in feats.items()}


with open(FEATURES_FILE, "rb") as f:
    data = pickle.load(f)

X = []
y = []

for sample in data:
    emb = sample["hidden_states"][LAYER].mean(axis=1).squeeze()

    feats = extract_features(sample["transcription"])
    if feats and FEATURE_TO_PLOT in feats:
        X.append(emb)
        y.append(feats[FEATURE_TO_PLOT])

X = np.array(X)
y = np.array(y)

pca = PCA(n_components=2)
X_2d = pca.fit_transform(X)

clf = LogisticRegression()
clf.fit(X_2d, y)

plt.figure()

for label in np.unique(y):
    idx = y == label
    plt.scatter(X_2d[idx, 0], X_2d[idx, 1], label=label)

x_min, x_max = X_2d[:, 0].min(), X_2d[:, 0].max()
y_min, y_max = X_2d[:, 1].min(), X_2d[:, 1].max()

xx, yy = np.meshgrid(
    np.linspace(x_min, x_max, 100),
    np.linspace(y_min, y_max, 100)
)

Z = clf.predict(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)

plt.contour(xx, yy, Z, alpha=0.3)

plt.title(f"Linear Separability of {FEATURE_TO_PLOT}")
plt.xlabel("PCA Component 1")
plt.ylabel("PCA Component 2")
plt.legend()

plt.show()

: 